In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# === Step 1: Load ITO Descriptions Recursively ===
def load_ito_descriptions(folder_path):
    descriptions = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                if file.endswith('.csv'):
                    df = pd.read_csv(file_path)
                elif file.endswith('.xlsx'):
                    df = pd.read_excel(file_path)
                else:
                    continue

                if 'description' in df.columns:
                    desc = df['description'].dropna().astype(str).tolist()
                    descriptions.extend(desc)
            except Exception as e:
                print(f"⚠ Failed to read {file_path}: {e}")
    return descriptions

# === Step 2: TF-IDF Vectorizer ===
def vectorize_text(corpus):
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    X = vectorizer.fit_transform(corpus).toarray()
    return X, vectorizer

# === Step 3: Train Autoencoder ===
def train_autoencoder(X):
    input_dim = X.shape[1]
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation="relu")(input_layer)
    encoded = Dense(32, activation="relu")(encoded)
    decoded = Dense(64, activation="relu")(encoded)
    decoded = Dense(input_dim, activation="sigmoid")(decoded)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer=Adam(0.001), loss='mse')
    autoencoder.fit(X, X, epochs=100, batch_size=8, verbose=1)

    reconstructions = autoencoder.predict(X)
    mse = np.mean(np.power(X - reconstructions, 2), axis=1)
    threshold = np.percentile(mse, 95)

    return autoencoder, threshold

# === Step 4: Predict on Test Data ===
def predict_test_file(test_file_path, vectorizer, model, threshold):
    if test_file_path.endswith('.csv'):
        test_df = pd.read_csv(test_file_path)
    elif test_file_path.endswith('.xlsx'):
        test_df = pd.read_excel(test_file_path)
    else:
        raise Exception("Unsupported test file format")

    if 'Description' not in test_df.columns:
        raise Exception("No 'description' column in test file")

    test_df['Description'] = test_df['Description'].astype(str)
    X_test = vectorizer.transform(test_df['Description']).toarray()

    reconstructions = model.predict(X_test)
    mse = np.mean(np.power(X_test - reconstructions, 2), axis=1)
    predictions = ['ITO' if e <= threshold else 'Non-ITO' for e in mse]
    test_df['predicted_label'] = predictions

    # Save to separate files
    test_df[test_df['predicted_label'] == 'ITO'].to_csv("ito_predictions.csv", index=False)
    test_df[test_df['predicted_label'] == 'Non-ITO'].to_csv("non_ito_predictions.csv", index=False)

    print("✅ Saved 'ito_predictions.csv' and 'non_ito_predictions.csv'")

# === Main ===

# 🔁 Replace with your actual paths
train_folder_path = "/path/to/ito/folder"
test_file_path = "/content/sample_tickets.csv"

# Load and train
ito_descriptions = load_ito_descriptions(train_folder_path)
X, vectorizer = vectorize_text(ito_descriptions)
model, threshold = train_autoencoder(X)

# Predict and save results
predict_test_file(test_file_path, vectorizer, model, threshold)

In [1]:
pip install tensorflow

In [7]:

import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# === Step 1: Load ITO Descriptions Recursively ===
def load_ito_descriptions(folder_path):
    descriptions = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                if file.endswith('.csv'):
                    df = pd.read_csv(file_path)
                elif file.endswith('.xlsx'):
                    df = pd.read_excel(file_path)
                else:
                    continue

                if 'Description' in df.columns:
                    desc = df['Description'].dropna().astype(str).tolist()
                    descriptions.extend(desc)
            except Exception as e:
                print(f"⚠ Failed to read {file_path}: {e}")
    return descriptions

# === Step 2: TF-IDF Vectorizer ===
def vectorize_text(corpus):
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    X = vectorizer.fit_transform(corpus).toarray()
    return X, vectorizer

# === Step 3: Train Autoencoder ===
def train_autoencoder(X):
    input_dim = X.shape[1]
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation="relu")(input_layer)
    encoded = Dense(32, activation="relu")(encoded)
    decoded = Dense(64, activation="relu")(encoded)
    decoded = Dense(input_dim, activation="sigmoid")(decoded)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer=Adam(0.001), loss='mse')
    autoencoder.fit(X, X, epochs=10, batch_size=8, verbose=1)

    reconstructions = autoencoder.predict(X)
    mse = np.mean(np.power(X - reconstructions, 2), axis=1)
    threshold = np.percentile(mse, 95)

    return autoencoder, threshold

# === Step 4: Predict on Test Data ===
def predict_test_file(test_file_path, vectorizer, model, threshold):
    if test_file_path.endswith('.csv'):
        test_df = pd.read_csv(test_file_path)
    elif test_file_path.endswith('.xlsx'):
        test_df = pd.read_excel(test_file_path)
    else:
        raise Exception("Unsupported test file format")

    if 'Document' not in test_df.columns:
        raise Exception("No 'description' column in test file")

    test_df['Document'] = test_df['Document'].astype(str)
    X_test = vectorizer.transform(test_df['Document']).toarray()

    reconstructions = model.predict(X_test)
    mse = np.mean(np.power(X_test - reconstructions, 2), axis=1)
    predictions = ['ITO' if e <= threshold else 'Non-ITO' for e in mse]
    test_df['predicted_label'] = predictions

    # Save to separate files
    test_df[test_df['predicted_label'] == 'ITO'].to_csv("ito_predictions.csv", index=False)
    test_df[test_df['predicted_label'] == 'Non-ITO'].to_csv("non_ito_predictions.csv", index=False)

    print("✅ Saved 'ito_predictions.csv' and 'non_ito_predictions.csv'")

# === Main ===

# 🔁 Replace with your actual paths
train_folder_path = "/content/drive/MyDrive/Usecases"
test_file_path = "/content/regex_ito_tickets.csv"

# Load and train
ito_descriptions = load_ito_descriptions(train_folder_path)
X, vectorizer = vectorize_text(ito_descriptions)
model, threshold = train_autoencoder(X)

# Predict and save results
predict_test_file(test_file_path, vectorizer, model, threshold)

⚠ Failed to read /content/drive/MyDrive/Usecases/Usecases/SQL Server Management Studio.csv: 'utf-8' codec can't decode byte 0x90 in position 22: invalid start byte
⚠ Failed to read /content/drive/MyDrive/Usecases/Usecases/New Relic.csv: 'utf-8' codec can't decode byte 0x90 in position 22: invalid start byte
⚠ Failed to read /content/drive/MyDrive/Usecases/Usecases/AppDynamics.csv: 'utf-8' codec can't decode byte 0x90 in position 22: invalid start byte
⚠ Failed to read /content/drive/MyDrive/Usecases/Usecases/Microsoft Performance Monitor.csv: 'utf-8' codec can't decode byte 0x90 in position 22: invalid start byte
⚠ Failed to read /content/drive/MyDrive/Usecases/Usecases/SolarWinds Server & Application Monitor.csv: 'utf-8' codec can't decode byte 0x90 in position 22: invalid start byte
⚠ Failed to read /content/drive/MyDrive/Usecases/Usecases/TS_ON/Microsoft SharePoint.csv: 'utf-8' codec can't decode byte 0x90 in position 22: invalid start byte
Epoch 1/10
1780/1780 ━━━━━━━━━━━━━━━━━━━━ 

In [8]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.neighbors import LocalOutlierFactor
from transformers import BertTokenizer, BertModel
import torch

# === Load BERT ===
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")
bert_model.eval()

# === BERT Embedding Function ===
def bert_embed(texts):
    embeddings = []
    for text in tqdm(texts, desc="Embedding texts"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        with torch.no_grad():
            outputs = bert_model(**inputs)
        mean_vec = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        embeddings.append(mean_vec)
    return np.array(embeddings)

# === Load ITO Descriptions from Folder ===
def load_ito_descriptions(folder_path):
    descriptions = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            path = os.path.join(root, file)
            try:
                if file.endswith(".csv"):
                    df = pd.read_csv(path, encoding='utf-8')
                elif file.endswith(".xlsx"):
                    df = pd.read_excel(path)
                else:
                    continue
                if 'Description' in df.columns:
                    descriptions.extend(df['Description'].dropna().astype(str).tolist())
            except Exception as e:
                print(f"⚠ Failed to read {file}: {e}")
    return descriptions

# === Load Test Descriptions ===
def load_test_descriptions(file_path):
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_path.endswith(".xlsx"):
        df = pd.read_excel(file_path)
    else:
        raise Exception("Unsupported file type")

    if 'Document' not in df.columns:
        raise Exception("No 'description' column found")

    df['Document'] = df['Document'].astype(str)
    return df

# === Train LOF Model on ITO ===
def train_lof_on_ito(ito_descriptions):
    ito_vectors = bert_embed(ito_descriptions)
    lof = LocalOutlierFactor(n_neighbors=5, novelty=True)
    lof.fit(ito_vectors)
    return lof

# === Train KMeans Model on ITO ===
def train_kmeans_on_ito(ito_descriptions):
    ito_vectors = bert_embed(ito_descriptions)
    kmeans = KMeans(n_clusters=1, random_state=42)  # Only one cluster: ITO
    kmeans.fit(ito_vectors)
    return kmeans

# === Predict on Test Data ===
def predict_on_test(test_df, lof_model, kmeans_model):
    test_vectors = bert_embed(test_df['Document'].tolist())

    # LOF predictions
    lof_scores = lof_model.decision_function(test_vectors)
    lof_predictions = ["ITO" if s > 0 else "Non-ITO" for s in lof_scores]

    # KMeans predictions (based on distance from center)
    distances = np.linalg.norm(test_vectors - kmeans_model.cluster_centers_[0], axis=1)
    threshold = np.percentile(distances, 95)
    kmeans_predictions = ["ITO" if d <= threshold else "Non-ITO" for d in distances]

    test_df["lof_prediction"] = lof_predictions
    test_df["kmeans_prediction"] = kmeans_predictions

    test_df[test_df["lof_prediction"] == "ITO"].to_csv("ito_lof_test.csv", index=False)
    test_df[test_df["lof_prediction"] == "Non-ITO"].to_csv("non_ito_lof_test.csv", index=False)

    test_df[test_df["kmeans_prediction"] == "ITO"].to_csv("ito_kmeans_test.csv", index=False)
    test_df[test_df["kmeans_prediction"] == "Non-ITO"].to_csv("non_ito_kmeans_test.csv", index=False)

    print("✅ Test predictions saved to CSVs")
    return test_df

# === Main ===
train_folder_path = "/content/drive/MyDrive/Usecases"
test_file_path = "/content/regex_ito_tickets.csv"

# Load and train
ito_descriptions = load_ito_descriptions(train_folder_path)
lof_model = train_lof_on_ito(ito_descriptions)
kmeans_model = train_kmeans_on_ito(ito_descriptions)

# Load and predict
test_df = load_test_descriptions(test_file_path)
predicted_df = predict_on_test(test_df, lof_model, kmeans_model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

⚠ Failed to read SQL Server Management Studio.csv: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2

⚠ Failed to read New Relic.csv: Error tokenizing data. C error: Expected 1 fields in line 6, saw 5

⚠ Failed to read AppDynamics.csv: Error tokenizing data. C error: Expected 1 fields in line 5, saw 3

⚠ Failed to read Microsoft Performance Monitor.csv: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2

⚠ Failed to read SolarWinds Server & Application Monitor.csv: Error tokenizing data. C error: Expected 1 fields in line 6, saw 2

⚠ Failed to read Microsoft SharePoint.csv: Error tokenizing data. C error: Expected 1 fields in line 11, saw 2



Embedding texts: 100%|██████████| 158/158 [00:17<00:00,  9.19it/s]


✅ Test predictions saved to CSVs
